In [2]:
import pandas as pd
import numpy as np

# Mock 20 days of Nifty data
data = {
    'Close': [24000, 24050, 24100, 24080, 24150, 24200, 24250, 24280, 24320, 24350,
              24400, 24450, 24480, 24500, 24550, 24580, 24600, 24650, 24700, 24750],
    'High': [24050, 24100, 24150, 24120, 24200, 24250, 24300, 24330, 24370, 24400,
             24450, 24500, 24530, 24550, 24600, 24630, 24650, 24700, 24750, 24800],
    'Low': [23950, 24000, 24050, 24030, 24100, 24150, 24200, 24230, 24270, 24300,
            24350, 24400, 24430, 24450, 24500, 24530, 24550, 24600, 24650, 24700],
    'Volume': [100, 110, 120, 95, 130, 140, 150, 145, 155, 160,
               170, 180, 175, 185, 190, 195, 200, 210, 220, 230]
}

df = pd.DataFrame(data)
print("Sample Data (last 5 rows):")
df.tail()

Sample Data (last 5 rows):


,Close,High,Low,Volume
15,24580,24630,24530,195
16,24600,24650,24550,200
17,24650,24700,24600,210
18,24700,24750,24650,220
19,24750,24800,24700,230


In [3]:
def calculate_rsi(series, period=14):
    """Calculate RSI with Wilder's smoothing"""
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    # Wilder's smoothing
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = calculate_rsi(df['Close'], period=14)
df['prev_RSI'] = df['RSI'].shift(1)

print("RSI Calculation (last 5 rows):")
df[['Close', 'RSI', 'prev_RSI']].tail()

RSI Calculation (last 5 rows):


,Close,RSI,prev_RSI
15,24580,97.805343,97.614224
16,24600,97.924700,97.805343
17,24650,98.189759,97.924700
18,24700,98.408643,98.189759
19,24750,98.591988,98.408643


In [4]:
df['MA20'] = df['Close'].rolling(20).mean()
df['EMA200'] = df['Close'].ewm(span=200).mean()  # Will have NaN initially
df['MA20_5d_ago'] = df['MA20'].shift(5)
df['Volume_MA20'] = df['Volume'].rolling(20).mean()

print("Moving Averages (last 5 rows):")
df[['Close', 'MA20', 'EMA200', 'MA20_5d_ago']].tail()

Moving Averages (last 5 rows):


,Close,MA20,EMA200,MA20_5d_ago
15,24580,NaN,24304.569387,NaN
16,24600,NaN,24323.372484,NaN
17,24650,NaN,24343.101765,NaN
18,24700,NaN,24363.624081,NaN
19,24750,24372.0,24384.832905,NaN


In [5]:
# For demo purposes, mock ADX values (real calculation is complex)
# In production, you'll use the calculate_adx function from your script
df['ADX'] = [15, 16, 17, 18, 19, 20, 21, 22, 23, 24,
             25, 26, 27, 26, 25, 24, 25, 26, 27, 28]
df['prev_ADX'] = df['ADX'].shift(1)
df['ADX_slope'] = df['ADX'].diff()

print("ADX Momentum (last 5 rows):")
df[['ADX', 'prev_ADX', 'ADX_slope']].tail()

ADX Momentum (last 5 rows):


,ADX,prev_ADX,ADX_slope
15,24,25.0,-1.0
16,25,24.0,1.0
17,26,25.0,1.0
18,27,26.0,1.0
19,28,27.0,1.0


In [6]:
def calculate_gss_v3(row):
    """GSS v3.0 with Leading Indicators"""
    score = 0

    # Factor 1: Long-term Anchor (15 pts)
    if row['Close'] > row['EMA200']:
        score += 15

    # Factor 2: MA20 Slope (20 pts) - Relaxed
    if pd.notna(row['MA20_5d_ago']):
        ma_slope = ((row['MA20'] - row['MA20_5d_ago']) / row['MA20_5d_ago']) * 100
        if ma_slope > 0.05:  # Relaxed from 0.1%
            score += 20

    # Factor 3A: ADX Strength (15 pts)
    if row['ADX'] > 20:  # Lowered from 25
        score += 15

    # Factor 3B: ADX Acceleration - LEADING (15 pts)
    if row['ADX'] > row['prev_ADX'] and row['ADX'] > 15:
        score += 15

    # Factor 4A: RSI Above 50 - LEADING (10 pts)
    if row['RSI'] > 50:
        score += 10

    # Factor 4B: RSI Rising - LEADING (10 pts)
    if row['RSI'] > row['prev_RSI']:
        score += 10

    # Factor 5: Price Proximity (15 pts) - Relaxed
    if pd.notna(row['MA20']):
        dist_pct = (row['Close'] - row['MA20']) / row['MA20'] * 100
        if 0 < dist_pct <= 3:  # Relaxed from 2%
            score += 15

    return score

# Apply to last row (index 19)
df['GSS_Score_v3'] = df.apply(calculate_gss_v3, axis=1)

print("GSS v3.0 Scores (last 5 rows):")
df[['Close', 'RSI', 'ADX', 'ADX_slope', 'GSS_Score_v3']].tail()

GSS v3.0 Scores (last 5 rows):


,Close,RSI,ADX,ADX_slope,GSS_Score_v3
15,24580,97.805343,24,-1.0,50
16,24600,97.924700,25,1.0,65
17,24650,98.189759,26,1.0,65
18,24700,98.408643,27,1.0,65
19,24750,98.591988,28,1.0,80


In [7]:
def calculate_gss_old(row):
    """OLD GSS v2.0 (Current conservative version)"""
    score = 0

    if row['Close'] > row['EMA200']:
        score += 20

    if pd.notna(row['MA20_5d_ago']):
        ma_slope = ((row['MA20'] - row['MA20_5d_ago']) / row['MA20_5d_ago']) * 100
        if ma_slope > 0.1:  # Strict
            score += 30

    if row['ADX'] > 25:  # Strict
        score += 30
    elif row['ADX'] > row['prev_ADX'] and row['ADX'] > 15:
        score += 20

    if pd.notna(row['MA20']):
        dist_pct = (row['Close'] - row['MA20']) / row['MA20'] * 100
        if 0 < dist_pct <= 2:  # Strict
            score += 20

    return score

df['GSS_Score_OLD'] = df.apply(calculate_gss_old, axis=1)

print("\n📊 COMPARISON (last 5 rows):")
print(df[['Close', 'RSI', 'ADX', 'ADX_slope', 'GSS_Score_OLD', 'GSS_Score_v3']].tail())
print(f"\n⚡ Score Improvement: {df['GSS_Score_v3'].iloc[-1] - df['GSS_Score_OLD'].iloc[-1]} points")


📊 COMPARISON (last 5 rows):
    Close        RSI  ADX  ADX_slope  GSS_Score_OLD  GSS_Score_v3
15  24580  97.805343   24       -1.0             20            50
16  24600  97.924700   25        1.0             40            65
17  24650  98.189759   26        1.0             50            65
18  24700  98.408643   27        1.0             50            65
19  24750  98.591988   28        1.0             70            80

⚡ Score Improvement: 10 points


In [8]:
def map_to_regime_v3(score, volume, volume_ma20, adx_slope):
    """Dynamic volume threshold based on ADX momentum"""
    volume_ratio = volume / volume_ma20

    # Adjust threshold based on momentum
    if adx_slope > 5:  # Strong acceleration
        volume_threshold = 1.1
        label = "🚀 HIGH MOMENTUM"
    else:
        volume_threshold = 1.2
        label = "📊 STANDARD"

    volume_confirmed = volume_ratio > volume_threshold

    if score >= 70 and volume_confirmed:
        regime = "BULL"
    elif score >= 30:
        regime = "SIDEWAYS"
    else:
        regime = "BEAR"

    return regime, volume_threshold, label

# Test on last row
last_row = df.iloc[-1]
regime, threshold, label = map_to_regime_v3(
    last_row['GSS_Score_v3'],
    last_row['Volume'],
    last_row['Volume_MA20'],
    last_row['ADX_slope']
)

print(f"\n🎯 REGIME PREDICTION:")
print(f"GSS Score: {last_row['GSS_Score_v3']}")
print(f"Volume Ratio: {last_row['Volume']/last_row['Volume_MA20']:.2f}x")
print(f"Volume Threshold: {threshold}x ({label})")
print(f"Predicted Regime: {regime}")


🎯 REGIME PREDICTION:
GSS Score: 80.0
Volume Ratio: 1.41x
Volume Threshold: 1.2x (📊 STANDARD)
Predicted Regime: BULL
